# Predicting for the House Price Challenge:  Andy Zhu and Sofya Kurilina


Started by importing the libraries we needed and loading the training and testing datasets. The training set contains 1,200 houses with their actual sale prices, while the test set has 260 houses that we need to predict prices for.

In [1]:
import os
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict

!rm -f train*.csv test*.csv sample*.csv
uploaded = files.upload()

Saving train.csv to train.csv
Saving test.csv to test.csv


In [2]:
df = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print("Training data:", df.shape)
print("Testing data:", test.shape)
print("CSV files in Colab:", [f for f in os.listdir() if f.endswith('.csv')])

Training data: (1095, 81)
Testing data: (365, 80)
CSV files in Colab: ['train.csv', 'test.csv']


We noticed two very large houses sold for much less than their size suggest and they throw the model off, so we droped them from the training data.

In [3]:
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))].reset_index(drop=True)
print("Training houses remaining:", len(df))

Training houses remaining: 1093


Total square footage, total bathrooms, house age, years since remodel, and quality x size (square footage matters more in a higher-quality house) are made numerical. then make other data a string

In [4]:
def add_features(d):
    d = d.copy()
    d['TotalSF'] = d['TotalBsmtSF'] + d['1stFlrSF'] + d['2ndFlrSF']
    d['TotBath'] = (d['FullBath'] + 0.5 * d['HalfBath']
                    + d['BsmtFullBath'] + 0.5 * d['BsmtHalfBath'])
    d['Age'] = (d['YrSold'] - d['YearBuilt']).clip(lower=0)
    d['RemodAge'] = (d['YrSold'] - d['YearRemodAdd']).clip(lower=0)
    d['QualSF'] = d['OverallQual'] * d['GrLivArea']
    d['PorchSF'] = (d['OpenPorchSF'] + d['EnclosedPorch'] + d['3SsnPorch']
                    + d['ScreenPorch'] + d['WoodDeckSF'])
    d['MSSubClass'] = d['MSSubClass'].astype(str)
    d['MoSold'] = d['MoSold'].astype(str)
    return d

df = add_features(df)
test = add_features(test)

House prices and areas have a few very large values. Taking the log keeps those from overpowering and skwewing the model.

In [5]:
y = df['SalePrice']
y_log = np.log1p(y)

nums = df.select_dtypes('number').columns.drop(['Id', 'SalePrice']).tolist()

skewed = [col for col in nums if df[col].min() >= 0 and df[col].skew() > 0.75]
df[skewed] = np.log1p(df[skewed].clip(lower=0))
test[skewed] = np.log1p(test[skewed].clip(lower=0))
print("Log-transformed columns:", len(skewed))

Log-transformed columns: 21


we sort the columns into groups. A missing value means the house has no such feature, so it becomes 'None' (the lowest level). The remaining text columns use OneHotEncoder. Number columns get the mean filled in for missing values.

In [6]:
ord_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
            'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
vals = [['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']]

cats = df.select_dtypes(exclude='number').columns.tolist()
ohe_cols = [col for col in cats if col not in ord_cols]

print("Number columns:", len(nums))
print("Ordinal columns:", len(ord_cols))
print("One-hot columns:", len(ohe_cols))

Number columns: 40
Ordinal columns: 10
One-hot columns: 35


In [7]:
cat_imputer = SimpleImputer(strategy='constant', fill_value='None')
ord_imputer = SimpleImputer(strategy='constant', fill_value='None')
ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ord_enc = OrdinalEncoder(categories=vals * len(ord_cols),
                         handle_unknown='use_encoded_value', unknown_value=-1)

ohe_pipe = Pipeline([('impute', cat_imputer), ('ohe', ohe)])
ord_pipe = Pipeline([('impute', ord_imputer), ('ord', ord_enc)])

transformer = make_column_transformer((ohe_pipe, ohe_cols),
                                      (ord_pipe, ord_cols),
                                      (SimpleImputer(), nums))

def make_model(alpha):
    return Pipeline([('transform', transformer),
                     ('scale', StandardScaler()),
                     ('model', Ridge(alpha=alpha))])

X = df[nums + ord_cols + ohe_cols]
X_test = test[X.columns.tolist()]

using alpha with cross-validation, we tested on 5 different parts of the training data

In [8]:
def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_alpha, best_score = None, float('inf')
for alpha in [1, 3, 5, 10, 20, 30, 50]:
    pred_log = cross_val_predict(make_model(alpha), X, y_log, cv=kf)
    score = rmse(y, np.expm1(pred_log))
    print(f"alpha = {alpha:>3}:  RMSE = ${score:,.2f}")
    if score < best_score:
        best_alpha, best_score = alpha, score

print(f"\nBest alpha: {best_alpha}  (RMSE ${best_score:,.2f})")

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [6, 16, 17] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [15, 16, 17, 23, 25, 26, 33] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 15, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [9, 16, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/

alpha =   1:  RMSE = $24,559.25


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [15, 16, 17, 23, 25, 26, 33] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 15, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [9, 16, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


alpha =   3:  RMSE = $24,252.67


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [6, 16, 17] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [15, 16, 17, 23, 25, 26, 33] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 15, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [9, 16, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/

alpha =   5:  RMSE = $24,113.99


... (5 outputs left out)

alpha =  30:  RMSE = $23,507.72


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [6, 16, 17] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [15, 16, 17, 23, 25, 26, 33] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 15, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [9, 16, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


alpha =  50:  RMSE = $23,356.41

Best alpha: 50  (RMSE $23,356.41)


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [9]:
model_pipeline = make_model(best_alpha)
model_pipeline.fit(X, y_log)

preds = np.expm1(model_pipeline.predict(X_test))

pdf = pd.DataFrame(preds, columns=['SalePrice'])
pdf['Id'] = test.index   # Id is case sensitive
pdf = pdf[['Id', 'SalePrice']]
pdf.to_csv('submission.csv', index=False)
pdf.head()

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 15, 25] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,Id,SalePrice
0,0,146141.216738
1,1,334121.436997
2,2,104335.762630
3,3,165364.042012
4,4,315634.555558


In [10]:
print(pdf.head())
print("Rows:", len(pdf), "| Id goes from", pdf['Id'].min(), "to", pdf['Id'].max())

assert len(pdf) == len(test)
assert pdf['Id'].min() == 0
assert pdf['SalePrice'].notna().all()

if os.path.exists('sample_soln.csv'):
    sample = pd.read_csv('sample_soln.csv')
    print("Ids match sample_soln.csv:", (sample['Id'].values == pdf['Id'].values).all())

   Id      SalePrice
0   0  146141.216738
1   1  334121.436997
2   2  104335.762630
3   3  165364.042012
4   4  315634.555558
Rows: 365 | Id goes from 0 to 364


In [11]:
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>